<a href="https://colab.research.google.com/github/Luaalmed/Aula-02-Intelig-ncia-Artificial/blob/main/AG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Algoritmo Genético - *Knapsack Problem*

In [1]:
!pip install pyeasyga pyswarms


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.4 MB/s eta 0:00:00
  Created wheel for pyeasyga: filename=pyeasyga-0.3.1-py2.py3-none-any.whl size=6784 sha256=961880677f21e8efe1899587f4a05df60af55ba7ae5dd82a14bf09e09672fa5a
  Stored in directory: /root/.cache/pip/wheels/5b/cb/8f/1f54efc0a60a5c5ea25349372a2c108415b214375b0aa276f7
Successfully built pyeasyga


In [2]:
from pyeasyga import pyeasyga
import random

data = [{'name': 'verde', 'value': 4, 'weight': 12},
        {'name': 'cinza', 'value': 2, 'weight': 1},
        {'name': 'amarelo', 'value': 10, 'weight': 4},
        {'name': 'laranja', 'value': 1, 'weight': 1},
        {'name': 'azul', 'value': 2, 'weight': 2}]

tamanho_populacao = 20
geracoes = 50

ga = pyeasyga.GeneticAlgorithm(data,
                               population_size = tamanho_populacao,
                               generations = geracoes,
                               crossover_probability = 0.9,
                               mutation_probability = 0.3,
                               elitism = True,
                               maximise_fitness = True)

def my_create_individual(data):
    return [random.randint(0, 15) for _ in range(len(data))]
ga.create_individual = my_create_individual

def aptidao(individual, data):
    dinheiro = 0
    peso = 0
    for quantidade, caixa in zip(individual, data):
        dinheiro += quantidade * caixa['value']
        peso += quantidade * caixa['weight']
    if peso > 15:
        return 0
    return dinheiro
ga.fitness_function = aptidao

def crossover(parent_1, parent_2):
    corte = random.randrange(1, len(parent_1))
    child_1 = parent_1[:corte] + parent_2[corte:]
    child_2 = parent_2[:corte] + parent_1[corte:]
    return child_1, child_2
ga.crossover_function = crossover

def my_mutation(individual):
    posicao = random.randrange(len(individual))
    individual[posicao] = random.randint(0, 15)
ga.mutate_function = my_mutation

def my_selection(population):
    competidores = random.sample(population, 3)
    return max(competidores, key=lambda ind: ind.fitness)
ga.selection_function = my_selection

ga.run()

print("Melhor solução GA (Mochila):", ga.best_individual())

Melhor solução GA (Mochila): (0, [6, 15, 13, 8, 4])


In [3]:
import pyswarms as ps
import numpy as np

data = [{'name': 'verde', 'value': 4, 'weight': 12},
        {'name': 'cinza', 'value': 2, 'weight': 1},
        {'name': 'amarelo', 'value': 10, 'weight': 4},
        {'name': 'laranja', 'value': 1, 'weight': 1},
        {'name': 'azul', 'value': 2, 'weight': 2}]

def aptidao_pso(enxame, data=data):
    resultados = []
    for particula in enxame:
        individual = np.round(particula).astype(int)
        individual = np.maximum(0, individual)

        dinheiro, peso = 0, 0
        for quantidade, caixa in zip(individual, data):
            dinheiro += quantidade * caixa['value']
            peso += quantidade * caixa['weight']

        if peso > 15:
            dinheiro = 0

        resultados.append(-dinheiro)

    return np.array(resultados)

bounds = (np.zeros(5), np.full(5, 15))
options = {'c1': 1.5, 'c2': 1.5, 'w': 0.7}

pso = ps.single.GlobalBestPSO(n_particles=30, dimensions=5, options=options, bounds=bounds)

melhor_custo, melhor_individuo = pso.optimize(aptidao_pso, iters=50)

individuo_final = np.round(melhor_individuo).astype(int)
print("Melhor solução PSO (Quantidades):", individuo_final)
print("Dinheiro obtido:", -melhor_custo)


2026-08-29 21:57:43,985 - pyswarms.single.global_best - INFO - Optimize for 50 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.7}
pyswarms.single.global_best: 100%|██████████|50/50, best_cost=0
2026-08-29 21:57:44,105 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0, best pos: [ 6.34708796 10.77466107 13.502563   12.65785817  7.24968544]


Melhor solução PSO (Quantidades): [ 6 11 14 13  7]
Dinheiro obtido: -0.0


# Exemplo inicial para o Laboratório

## Com repetições

#### É necessário alterar os métodos padrões da classe *GeneticAlgorithm*